# PyTorch Dynamic Quantization

This notebook trains a tiny MLP and then quantizes it to INT8.

Goals:
- train a simple baseline model
- apply dynamic quantization
- compare size, accuracy, and latency


In [ ]:
import io
import time

import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

try:
    from torch.ao.quantization import quantize_dynamic
except ImportError:
    from torch.quantization import quantize_dynamic

torch.manual_seed(42)

digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)


In [ ]:
class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return correct / total


def model_size_kb(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return len(buffer.getvalue()) / 1024


model = SmallMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

fp32_acc = evaluate(model)
fp32_size = model_size_kb(model)


In [ ]:
quantized_model = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
int8_acc = evaluate(quantized_model)
int8_size = model_size_kb(quantized_model)

sample_batch = X_test[:256]

def benchmark(model, runs=300):
    model.eval()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(runs):
            model(sample_batch)
    return (time.perf_counter() - start) * 1000 / runs


fp32_ms = benchmark(model)
int8_ms = benchmark(quantized_model)

pd.DataFrame(
    [
        {"model": "fp32", "accuracy": fp32_acc, "size_kb": fp32_size, "latency_ms": fp32_ms},
        {"model": "int8", "accuracy": int8_acc, "size_kb": int8_size, "latency_ms": int8_ms},
    ]
)
